In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Data Reading

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@storageaccnitin.dfs.core.windows.net/customers")
df.display()

In [0]:
df = df.drop("_rescued_data")
df.display()

In [0]:
df = df.withColumn("domain", split(col("email"),"@")[1])
df.display()

In [0]:
df.groupBy("domain").agg(count("customer_id").alias("COUNT")).sort(desc(col("COUNT"))).display()

In [0]:
df_gmail = df.filter(col("domain") == "gmail.com")
df_gmail.count()

In [0]:
df = df.withColumn("Full_Name", concat(col("first_name"),lit(' '),col("last_name"))).drop("first_name","last_name")
df.display()

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@storageaccnitin.dfs.core.windows.net/customers")

In [0]:
spark.read.format("delta").load("abfss://silver@storageaccnitin.dfs.core.windows.net/customers").display()

In [0]:
%sql
create table if not exists dbcatalog.silver.customers_silver 
using delta
location 'abfss://silver@storageaccnitin.dfs.core.windows.net/customers'

In [0]:
%sql
select * from dbcatalog.silver.customers_silver 